# Volume 2. Sequence Memory And LRI

Question: can a short trail of assemblies be stored, inspected, and cued back into activity?

This notebook focuses on process. Sequence memory is not presented as perfect recall; the trace shows how long-range inhibition can push activity away from the current attractor and into the next candidate.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from neural_assemblies.assembly_calculus import chance_overlap, overlap, sequence_memorize
from neural_assemblies.assembly_calculus.tracing import ordered_recall_trace
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import (
    animate_assembly_trace,
    plot_assemblies,
    plot_overlap_matrix,
    plot_recall_trace,
    plot_winner_turnover,
)


The trail is four cue labels in one SEQ area. Replaying the sequence strengthens bridges between consecutive assemblies.


In [ ]:
N = 5_000
K = 60
BETA = 0.10
STIMULI = ["morning", "coffee", "code", "notes"]

brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
for stimulus in STIMULI:
    brain.add_stimulus(stimulus, K)
brain.add_area("SEQ", N, K, BETA)

pd.DataFrame({
    "position": range(1, len(STIMULI) + 1),
    "stimulus": STIMULI,
    "area": "SEQ",
    "winners_per_assembly": K,
    "chance_overlap": chance_overlap(K, N),
})


In [ ]:
memorized = sequence_memorize(
    brain,
    STIMULI,
    "SEQ",
    rounds_per_step=10,
    repetitions=3,
    phase_b_ratio=0.5,
    beta_boost=0.25,
)

pd.DataFrame([
    {
        "position": index + 1,
        "stimulus": stimulus,
        "winner_count": len(assembly),
        "sample_winners": assembly.winners[:10].tolist(),
    }
    for index, (stimulus, assembly) in enumerate(zip(STIMULI, memorized))
])


In [ ]:
fig, axes = plot_assemblies(
    list(memorized),
    N,
    titles=[f"SEQ: {stimulus}" for stimulus in STIMULI],
    subtitles=[f"position {index}" for index in range(1, len(STIMULI) + 1)],
    marker_size=16,
)
plt.show()
plt.close(fig)


In [ ]:
ax, matrix = plot_overlap_matrix(list(memorized), labels=STIMULI, cmap="magma")
ax.set_title("Stored sequence assemblies")
plt.show()
plt.close(ax.figure)


Recall uses LRI: recently active winners are penalized, so self-projection is less likely to stay on the same assembly. The trace stops when it cycles, reaches `max_steps`, or leaves the known sequence neighborhood.


In [ ]:
brain.set_lri("SEQ", refractory_period=3, inhibition_strength=100.0)
recall_trace = ordered_recall_trace(
    brain,
    "SEQ",
    "morning",
    max_steps=8,
    known_assemblies=list(memorized),
    rounds_per_step=1,
)

pd.DataFrame(recall_trace.to_records())


In [ ]:
fig, animation = animate_assembly_trace(
    recall_trace,
    N,
    title="Cued recall with long-range inhibition",
    color="#7768ae",
    interval=850,
)
html = HTML(animation.to_jshtml())
plt.close(fig)
html


In [ ]:
recalled_labels = [f"step {step.round_index}" for step in recall_trace]
ax, recall_matrix = plot_recall_trace(
    recall_trace.assemblies,
    list(memorized),
    recalled_labels=recalled_labels,
    known_labels=STIMULI,
)
ax.set_title("Recall trace compared with stored sequence")
plt.show()
plt.close(ax.figure)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
plot_winner_turnover(recall_trace, ax=axes[0], title="Recall winner turnover")
axes[1].plot(
    [step.round_index for step in recall_trace],
    [max(overlap(step.assembly, known) for known in memorized) for step in recall_trace],
    marker="o",
    color="#7768ae",
)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("recall step")
axes[1].set_ylabel("best overlap with stored item")
axes[1].set_title("How close each recalled state is")
axes[1].grid(axis="y", alpha=0.25)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
plt.show()
plt.close(fig)


The lesson is the mechanism, not a guarantee: Hebbian bridges plus LRI can move activity along a stored trail, and the recall trace makes successes, early stops, and drift inspectable.
